# Pre-class: Monday Morning — Data Without Labels
**⏱ This pre-class notebook takes approximately 15 minutes.**

---

## Scenario: Monday — Marcus's New Brief

It's the start of week 5 at NorthStar Retail. Friday last week Sarah presented her tuned Gradient Boosting model. Marcus signed off on shipping it to production. Then he said:

> *"Now — what about the customers who DON'T churn but also don't log in for a year? Can we find natural CLUSTERS of customer behaviour without labels?"*

This Monday morning Sarah opens her familiar `northstar_customers.csv` — same 10,000 customers as L03/L04 — but **drops the `churned` column**. From this week's perspective, there is no target. There are just *features*. She has to find structure that wasn't labelled in advance.

By Friday she has to:
1. Reduce the data to something visualisable.
2. Group customers into natural segments she can name and present.
3. Flag a "watch list" of unusual customers worth investigating.

**By the end of this notebook you will be able to:**
- Describe NorthStar's customer features (without referring to `churned`)
- Spot correlations between features
- Identify rows that look "different" to you — a gut-feel pre-check for the Isolation Forest notebook

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — ready to explore")

## Load the data and drop the label

The dataset is the same one from L03/L04. The DIFFERENCE is that this week we're treating `churned` as if it doesn't exist — to simulate the real-world case where you genuinely don't have labels.

In [ ]:
df = pd.read_csv("data/northstar_customers.csv")

# Drop the target column — that's the whole point of unsupervised
features = df.drop(columns=["customer_id", "churned"])

print(f"Loaded: {len(df):,} customers × {features.shape[1]} features")
print(f"(Original target column 'churned' has been DROPPED for this week.)")
print()
features.head()

## Step 1 — What features do we have?

10 features. Most are numerical (age, tenure, spend, etc.). Two are categorical (region, subscription_tier). Two have missing values from L03 (`last_login_days_ago` and `avg_review_polarity`).

For most unsupervised methods, we'll need to handle missing values + scale numerics + encode categoricals before training. That's the same preprocessing pipeline we built in L03.

In [ ]:
# Quick summary of each column
print("Feature types:")
print(features.dtypes)
print()
print("Missing values:")
missing = features.isna().sum()
print(missing[missing > 0])
print()
print("Numerical summary (excluding missing):")
print(features.describe(include="number").round(2))

## Step 2 — Correlations between features

Before clustering, it's useful to see which features move together. Tightly correlated features carry redundant information — that's exactly what PCA will compress.

In [ ]:
# Correlation matrix for the numerical features
num_features = features.select_dtypes(include="number")
corr = num_features.corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()

print("Notable correlations:")
# Find the top absolute correlations (excluding diagonal)
corr_pairs = (
    corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
for (a, b), v in corr_pairs.head(5).items():
    sign = "+" if corr.loc[a, b] > 0 else "−"
    print(f"  {a:24s} ↔ {b:24s}  r = {sign}{abs(corr.loc[a, b]):.2f}")

### 💡 What you should notice

- **Correlations are mostly modest** — most features carry independent information.
- **Some features will probably correlate with the dropped `churned` column** in supervised analysis, but here we're just looking at how features relate to EACH OTHER.

The presence of correlated features tells us PCA will compress meaningfully — but only by a little. If everything were uncorrelated, PCA wouldn't help.

## Step 3 — Are there visible "groups"?

Without labels, can we see natural groupings by plotting pairs of features? Let's try a few combinations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Plot 1: tenure vs spend
axes[0].scatter(features["tenure_months"], features["avg_monthly_spend_gbp"],
                alpha=0.3, s=10, color="steelblue")
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Avg monthly spend (£)")
axes[0].set_title("tenure × spend")

# Plot 2: tenure vs returns
axes[1].scatter(features["tenure_months"], features["returns_per_purchase"],
                alpha=0.3, s=10, color="coral")
axes[1].set_xlabel("Tenure (months)")
axes[1].set_ylabel("Returns per purchase")
axes[1].set_title("tenure × returns")

# Plot 3: tickets vs reviews (handles NaN)
mask = features["avg_review_polarity"].notna()
axes[2].scatter(features.loc[mask, "support_tickets_quarter"],
                features.loc[mask, "avg_review_polarity"],
                alpha=0.3, s=10, color="seagreen")
axes[2].set_xlabel("Support tickets (quarter)")
axes[2].set_ylabel("Avg review polarity")
axes[2].set_title("tickets × reviews")

plt.tight_layout()
plt.show()

print("Eyeball question: in any plot, do you see distinct clusters? Or a smooth cloud?")

### 💡 What you should notice

- The plots show **smooth clouds**, not distinct clusters. That's normal — clusters in real data are RARELY obvious in any pair of features.
- The unsupervised toolkit's job is to find structure that's **not obvious to the eye** in 2D — by combining many features at once (PCA) or finding distance-based groupings (K-Means).

This is why pure visual inspection doesn't work as customer segmentation. We need the tools.

## Step 4 — Eyeball some "unusual" customers

Even without an algorithm, some customers look unusual. Let's pick a few extreme rows and look at them.

In [ ]:
# Customers with the longest tenure AND highest support tickets
unusual_1 = features.sort_values(["tenure_months", "support_tickets_quarter"],
                                  ascending=[False, False]).head(3)
print("Long-tenured customers with high support-ticket counts:")
print(unusual_1.to_string())
print()
# Customers with high returns AND high spend (paradoxical)
unusual_2 = features[
    (features["returns_per_purchase"] > 0.3) &
    (features["avg_monthly_spend_gbp"] > 200)
].head(3)
print("High-spend AND high-return customers (unusual combination):")
print(unusual_2.to_string())

### 💡 What you should notice

- **Unusual customers stand out on COMBINATIONS of features**, not on any one feature.
- Long tenure + lots of support tickets is unusual — most long-tenure customers don't need support.
- High spend + high returns is unusual — usually high spenders are satisfied.

These manual heuristics are what Isolation Forest will systematise — it finds anomalies across ALL features at once, without needing you to specify the unusual combinations.

## ✅ Section Summary

| What we did | What it tells us |
|---|---|
| **Dropped the `churned` label** | This week is genuinely unsupervised |
| **Inspected correlations** | Features are mostly independent — PCA will compress modestly |
| **Scatter-plotted feature pairs** | No obvious visual clusters — we need real algorithms |
| **Eyeballed unusual customers** | Anomalies show up on feature COMBINATIONS, not single features |

**Bring to class:**
1. Two features you'd expect to correlate (then we'll check if PCA agrees they're redundant).
2. One row from the head() that looked "different" to you (then we'll see if Isolation Forest agrees).
3. One question — anything from this notebook that didn't click.

---
**In class → Open `02_pca.ipynb` first.** That notebook builds the PCA dimensionality reduction.